In [28]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
TRAIN_EMB_PATH = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/merged data embeddings/new_data_dense_embeddings.npz"
VAL_EMB_PATH = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/validation data embeddings/validation_dense_embeddings.npz"
TEST_EMB_PATH = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/test data embeddings/test_dense_embeddings.npz"
TRAIN_JSON = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/train.json"
VAL_JSON = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/validation.json"
TEST_JSON = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/test.json"
SEMANTIC_MATCHES = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/semantic_matches.json"

label_map = {"False": 0, "True": 1, "Conflicting": 2}
reverse_label_map = {0: "False", 1: "True", 2: "Conflicting"}

# ============================================================
# CONFIG
# ============================================================
BATCH_SIZE = 16
MAX_ITERS = 100
LEARNING_RATE = 1e-3
EARLY_STOP_PATIENCE = 20
EMB_DIM = 1024
TRACE_MAX = 20          # max reasoning traces (claim handled separately now)
SCALAR_DIM = 7          # engineered verdict-list features
ATTN_HIDDEN = 256
COMBINED_DIM = EMB_DIM * 3 + SCALAR_DIM   # claim + attn_pooled + disagreement_std + scalars
HIDDEN_DIMS = [512, 256, 128]
OUTPUT_DIM = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"GPUs available: {torch.cuda.device_count()}")
print(f"Combined input dim: {COMBINED_DIM}")

# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def load_npz(path):
    return np.load(path)

def numpy_to_python(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: numpy_to_python(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [numpy_to_python(item) for item in obj]
    return obj

def split_and_pad(emb_array, max_traces=TRACE_MAX, emb_dim=EMB_DIM):
    """Row 0 = claim embedding. Rows 1: = reasoning trace embeddings (15 or 20).
    Returns claim_emb (D,), traces_padded (max_traces, D), mask (max_traces,)."""
    claim_emb = emb_array[0].astype(np.float32)
    traces = emb_array[1:].astype(np.float32)
    n = traces.shape[0]
    mask = np.zeros(max_traces, dtype=np.float32)
    if n < max_traces:
        pad = np.zeros((max_traces - n, emb_dim), dtype=np.float32)
        traces_padded = np.vstack([traces, pad])
        mask[:n] = 1.0
    else:
        traces_padded = traces[:max_traces]
        mask[:max_traces] = 1.0
    return claim_emb, traces_padded, mask

def verdict_features(verdict_list, max_traces=TRACE_MAX):
    """Engineered features from the trace-level verdict votes:
    class fractions, majority fraction, distinct-class ratio, entropy, count ratio."""
    counts = {0: 0, 1: 0, 2: 0}
    for v in verdict_list:
        if v in label_map:
            counts[label_map[v]] += 1
    total = sum(counts.values())
    if total == 0:
        return np.zeros(SCALAR_DIM, dtype=np.float32)

    frac_false = counts[0] / total
    frac_true = counts[1] / total
    frac_conf = counts[2] / total
    majority_frac = max(counts.values()) / total
    distinct_ratio = sum(1 for c in counts.values() if c > 0) / 3.0
    fracs = [frac_false, frac_true, frac_conf]
    entropy = -sum(p * np.log(p + 1e-9) for p in fracs if p > 0)
    num_traces_norm = min(total / max_traces, 1.0)

    return np.array([frac_false, frac_true, frac_conf, majority_frac,
                      distinct_ratio, entropy, num_traces_norm], dtype=np.float32)

# ============================================================
# LABEL / ITEM MAPPING (per dataset)
# ============================================================

def create_train_id_to_item():
    """new_id -> matched TRAIN_JSON item (gives label + Verdict_list)."""
    semantic_matches = load_json(SEMANTIC_MATCHES)
    train_data = load_json(TRAIN_JSON)
    claim_to_item = {item['claim']: item for item in train_data}
    new_id_to_item = {}
    for match in semantic_matches:
        new_claim = match['new_claim']
        if new_claim in claim_to_item:
            new_id_to_item[match['new_id']] = claim_to_item[new_claim]
    return new_id_to_item

def create_val_id_to_item():
    val_data = load_json(VAL_JSON)
    return {idx: item for idx, item in enumerate(val_data)}

def create_test_id_to_item():
    test_data = load_json(TEST_JSON)
    return {item['query_id']: item for item in test_data}

# ============================================================
# DATASETS
# ============================================================

class BaseDataset(Dataset):
    """Shared __getitem__ logic; subclasses set emb_data, id_to_item, ids, label_key."""
    label_key = 'label'

    def __getitem__(self, idx):
        id_ = self.ids[idx]
        emb_array = self.emb_data[f'id_{id_}']
        claim_emb, traces_padded, mask = split_and_pad(emb_array)

        item = self.id_to_item[id_]
        label = label_map[item[self.label_key]]
        verdict_list = item.get('Verdict_list', [])
        scalars = verdict_features(verdict_list)

        return (torch.from_numpy(claim_emb),
                torch.from_numpy(traces_padded),
                torch.from_numpy(mask),
                torch.from_numpy(scalars),
                torch.tensor(label, dtype=torch.long))

    def __len__(self):
        return len(self.ids)

class TrainDataset(BaseDataset):
    label_key = 'label'
    def __init__(self):
        self.emb_data = load_npz(TRAIN_EMB_PATH)
        self.id_to_item = create_train_id_to_item()
        all_ids = sorted(int(k.split('_')[1]) for k in self.emb_data.keys() if k.startswith('id_'))
        self.ids = [i for i in all_ids if i in self.id_to_item]
        print(f"TRAIN dataset size: {len(self.ids)}")

class ValDataset(BaseDataset):
    label_key = 'label'
    def __init__(self):
        self.emb_data = load_npz(VAL_EMB_PATH)
        self.id_to_item = create_val_id_to_item()
        self.ids = sorted(self.id_to_item.keys())
        print(f"VAL dataset size: {len(self.ids)}")

class TestDataset(BaseDataset):
    label_key = 'Label'
    def __init__(self):
        self.emb_data = load_npz(TEST_EMB_PATH)
        self.id_to_item = create_test_id_to_item()
        self.ids = sorted(self.id_to_item.keys())
        print(f"TEST dataset size: {len(self.ids)}")

# ============================================================
# MODEL: attention pooling + disagreement std + scalar feats + MLP
# ============================================================

class AttentionPool(nn.Module):
    def __init__(self, dim, hidden=ATTN_HIDDEN):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, traces, mask):
        # traces: (B,T,D), mask: (B,T) with 1=valid, 0=pad
        scores = self.attn(traces).squeeze(-1)              # (B,T)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=1)               # (B,T)
        pooled = torch.bmm(weights.unsqueeze(1), traces).squeeze(1)  # (B,D)
        return pooled

def masked_std(traces, mask, eps=1e-6):
    mask_f = mask.unsqueeze(-1)                               # (B,T,1)
    count = mask_f.sum(dim=1).clamp(min=1.0)                  # (B,1)
    mean = (traces * mask_f).sum(dim=1) / count               # (B,D)
    var = ((traces - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
    return torch.sqrt(var + eps)

class TraceMLP(nn.Module):
    def __init__(self, combined_dim, hidden_dims, output_dim):
        super().__init__()
        self.attn_pool = AttentionPool(EMB_DIM)

        layers = [nn.BatchNorm1d(combined_dim)]
        prev = combined_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, claim, traces, mask, scalars):
        pooled = self.attn_pool(traces, mask)
        disagreement = masked_std(traces, mask)
        x = torch.cat([claim, pooled, disagreement, scalars], dim=1)
        return self.mlp(x)

# ============================================================
# TRAINING
# ============================================================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for claim, traces, mask, scalars, labels in loader:
        claim, traces, mask, scalars, labels = (
            claim.to(device), traces.to(device), mask.to(device),
            scalars.to(device), labels.to(device)
        )
        optimizer.zero_grad()
        logits = model(claim, traces, mask, scalars)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for claim, traces, mask, scalars, labels in loader:
            claim, traces, mask, scalars, labels = (
                claim.to(device), traces.to(device), mask.to(device),
                scalars.to(device), labels.to(device)
            )
            logits = model(claim, traces, mask, scalars)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return avg_loss, acc, all_preds, all_labels

# ============================================================
# METRICS
# ============================================================

def generate_report(all_labels, all_preds, set_name="VAL"):
    all_labels = np.array(all_labels, dtype=int)
    all_preds = np.array(all_preds, dtype=int)
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2])

    print(f"\n{'='*70}\n{set_name} SET - DETAILED METRICS REPORT\n{'='*70}\n")
    print("CONFUSION MATRIX (rows=true, cols=pred, order=False/True/Conflicting):")
    print(cm)
    print()

    precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

    print(f"{'Class':<10}{'Label':<14}{'Precision':<12}{'Recall':<12}{'F1-Score':<12}")
    print("-" * 60)
    for c in range(3):
        print(f"{c:<10}{reverse_label_map[c]:<14}{precision[c]:<12.4f}{recall[c]:<12.4f}{f1[c]:<12.4f}")

    avg_p, avg_r, avg_f1 = float(np.mean(precision)), float(np.mean(recall)), float(np.mean(f1))
    print("-" * 60)
    print(f"{'MACRO AVG':<24}{avg_p:<12.4f}{avg_r:<12.4f}{avg_f1:<12.4f}\n")

    wp = float(precision_score(all_labels, all_preds, average='weighted', zero_division=0))
    wr = float(recall_score(all_labels, all_preds, average='weighted', zero_division=0))
    wf = float(f1_score(all_labels, all_preds, average='weighted', zero_division=0))
    print(f"Weighted Precision: {wp:.4f}")
    print(f"Weighted Recall:    {wr:.4f}")
    print(f"Weighted F1-Score:  {wf:.4f}\n")

    try:
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=list(reverse_label_map.values()),
                    yticklabels=list(reverse_label_map.values()))
        plt.title(f'{set_name} SET - Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(f'/kaggle/working/{set_name.lower()}_confusion_matrix.png', dpi=100, bbox_inches='tight')
        plt.close()
        print(f"Confusion matrix saved: /kaggle/working/{set_name.lower()}_confusion_matrix.png")
    except Exception as e:
        print(f"Warning: plot failed: {e}")

    return {
        'cm': cm.tolist(), 'precision': numpy_to_python(precision),
        'recall': numpy_to_python(recall), 'f1': numpy_to_python(f1),
        'avg_precision': avg_p, 'avg_recall': avg_r, 'avg_f1': avg_f1,
        'weighted_precision': wp, 'weighted_recall': wr, 'weighted_f1': wf
    }

# ============================================================
# MAIN
# ============================================================

def main():
    print("Loading datasets...")
    train_dataset = TrainDataset()
    val_dataset = ValDataset()
    test_dataset = TestDataset()

    train_labels_list = [label_map[train_dataset.id_to_item[i]['label']] for i in train_dataset.ids]
    counts = Counter(train_labels_list)
    total = sum(counts.values())
    weights = torch.tensor([total / (3 * counts[c]) for c in range(3)], dtype=torch.float32).to(DEVICE)
    print(f"Class counts: {dict(counts)}")
    print(f"Class weights (loss): {weights.cpu().tolist()}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    print("\nBuilding model...")
    model = TraceMLP(COMBINED_DIM, HIDDEN_DIMS, OUTPUT_DIM).to(DEVICE)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss(weight=weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    print("\nStarting training...")
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in range(MAX_ITERS):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        print(f"Epoch {epoch+1:3d}/{MAX_ITERS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict()
            print("  Best model saved")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

    if torch.cuda.device_count() > 1:
        model.module.load_state_dict(best_model_state)
    else:
        model.load_state_dict(best_model_state)

    print("\n" + "="*70 + "\nFINAL EVALUATION\n" + "="*70)

    _, _, train_preds, train_labels = evaluate(model, train_loader, criterion, DEVICE)
    train_metrics = generate_report(train_labels, train_preds, "TRAIN")

    _, _, val_preds, val_labels = evaluate(model, val_loader, criterion, DEVICE)
    val_metrics = generate_report(val_labels, val_preds, "VAL")

    _, _, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)
    test_metrics = generate_report(test_labels, test_preds, "TEST")

    results = {
        'train_metrics': train_metrics, 'val_metrics': val_metrics, 'test_metrics': test_metrics,
        'config': {
            'batch_size': BATCH_SIZE, 'max_iters': MAX_ITERS, 'learning_rate': LEARNING_RATE,
            'hidden_dims': HIDDEN_DIMS, 'combined_dim': COMBINED_DIM, 'output_dim': OUTPUT_DIM,
            'early_stop_patience': EARLY_STOP_PATIENCE, 'class_weights': weights.cpu().tolist(),
            'scalar_features': ['frac_false', 'frac_true', 'frac_conflicting', 'majority_frac',
                                 'distinct_ratio', 'entropy', 'num_traces_norm']
        }
    }
    with open('/kaggle/working/training_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("\nAll results saved to /kaggle/working/")

if __name__ == "__main__":
    main()

Device: cuda
GPUs available: 2
Combined input dim: 3079
Loading datasets...
TRAIN dataset size: 6400
VAL dataset size: 1600
TEST dataset size: 2558
Class counts: {2: 1508, 0: 3717, 1: 1175}
Class weights (loss): [0.5739395618438721, 1.8156027793884277, 1.4146772623062134]

Building model...
Using 2 GPUs

Starting training...
Epoch   1/100 | Train Loss: 1.0070 | Val Loss: 0.8416 | Val Acc: 0.5956
  Best model saved
Epoch   2/100 | Train Loss: 0.9315 | Val Loss: 0.7784 | Val Acc: 0.6456
  Best model saved
Epoch   3/100 | Train Loss: 0.8838 | Val Loss: 0.7731 | Val Acc: 0.6462
  Best model saved
Epoch   4/100 | Train Loss: 0.8459 | Val Loss: 0.8083 | Val Acc: 0.6194
Epoch   5/100 | Train Loss: 0.7963 | Val Loss: 0.8926 | Val Acc: 0.6219
Epoch   6/100 | Train Loss: 0.7670 | Val Loss: 0.9254 | Val Acc: 0.6206
Epoch   7/100 | Train Loss: 0.7356 | Val Loss: 1.2436 | Val Acc: 0.5813
Epoch   8/100 | Train Loss: 0.6195 | Val Loss: 1.6875 | Val Acc: 0.5756
Epoch   9/100 | Train Loss: 0.5673 | Val

In [29]:
print(0)

0
